### GraphRAG with Built-in LangChain

In [1]:
import os
import logging
from pathlib import Path
from dotenv import load_dotenv

os.environ['ANONYMIZED_TELEMETRY'] = 'False' 
logging.getLogger('httpx').setLevel(logging.WARNING)

from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain.chains import GraphQAChain
from langchain_community.graphs.networkx_graph import NetworkxEntityGraph
from langchain_experimental.graph_transformers import LLMGraphTransformer

In [2]:
load_dotenv()

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)


Configure LLMGraphTransformer

Component 1 — LLMGraphTransformer (Extraction)
* A LangChain class that extracts nodes (entities) and relationships from text using an LLM

**allowed_nodes — the types of entities allowed (Disease, Crop, Symptom, etc.)**

**allowed_relationships — the types of relations allowed (CAUSED_BY, AFFECTS, etc.)**

In [3]:
transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=['Disease', 'Crop', 'Symptom', 'Vector', 'Population', 'Control', 'Country'],
    allowed_relationships=['IS_A', 'CAUSED_BY', 'TRANSMITTED_BY', 'AFFECTS', 'HAS_SYMPTOM', 'CONTROL']
)

print('Transformer ready')
print(f'   Allowed nodes:         {len(transformer.allowed_nodes)} types')
print(f'   Allowed relationships: {len(transformer.allowed_relationships)} types')

Transformer ready
   Allowed nodes:         7 types
   Allowed relationships: 6 types


Load chunks from the vector store

In [4]:
# Load the domain-tagged vector store
persist_dir = r'C:\Users\USER\rag_course\chroma_db_domain'

embeddings = OpenAIEmbeddings()
vectorstore = Chroma(
    persist_directory=persist_dir,
    embedding_function=embeddings
)

# Get all chunks
data = vectorstore.get(include=['documents', 'metadatas'])
chunks_text = data['documents']
chunks_meta = data['metadatas']

print(f'Loaded {len(chunks_text)} chunks from the vector store')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Loaded 80 chunks from the vector store


Convert all chunks to LangChain Document objects

In [5]:
documents = [
    Document(page_content=text, metadata=meta)
    for text, meta in zip(chunks_text, chunks_meta)
]

print(f'Loaded {len(documents)} Document objects')

Loaded 80 Document objects


Extract triples with LLMGraphTransformer

In [6]:
# Convert all documents to graph documents
graph_documents = transformer.convert_to_graph_documents(documents)

# Summarize
total_nodes = sum(len(gd.nodes) for gd in graph_documents)
total_relationships = sum(len(gd.relationships) for gd in graph_documents)

print(f'Extracted {len(graph_documents)} graph documents')
print(f'Total node: {total_nodes}')
print(f'Total relationships: {total_relationships}')

Extracted 80 graph documents
Total node: 675
Total relationships: 521


In [7]:
graph_documents[:3]

[GraphDocument(nodes=[Node(id='Government', type='Population'), Node(id='Department Of Public Health', type='Control'), Node(id='Fmoh', type='Control'), Node(id='Cerebral Spinal Meningitis', type='Disease'), Node(id='Cholera', type='Disease'), Node(id='Measles', type='Disease'), Node(id='Lassa Fever', type='Disease'), Node(id='Yellow Fever', type='Disease'), Node(id='Diarrhoeal', type='Disease'), Node(id='Malaria', type='Disease'), Node(id='Plague', type='Disease'), Node(id='Tuberculosis', type='Disease'), Node(id='Pertussis', type='Disease'), Node(id='Onchocerciasis', type='Disease'), Node(id='Pneumonia', type='Disease'), Node(id='Hiv/Aids', type='Disease'), Node(id='Sti', type='Disease'), Node(id='Hepatitis B', type='Disease'), Node(id='National Public Health Agency', type='Control'), Node(id='Nigerians', type='Population')], relationships=[Relationship(source=Node(id='Department Of Public Health', type='Control'), target=Node(id='Fmoh', type='Control'), type='IS_A'), Relationship(so

Build the graph with NetworkxEntityGraph

In [8]:
# Create an empty NetworkxEntityGraph
graph = NetworkxEntityGraph()

# Add all graph documents to the graph
# Iterate over each GraphDocument and add nodes + edges
for gd in graph_documents:
    # Add all nodes from this document
    for node in gd.nodes:
        graph.add_node(node.id)
        
    # Add all relationships (edges) from this document
    for edge in gd.relationships:
        graph._graph.add_edge(
            edge.source.id,
            edge.target.id,
            relation=edge.type,
        )
    
# Summary
print('Graph built')
print(f'Number of nodes: {graph.get_number_of_nodes()}')
print(f'Number of tripples: {len(graph.get_triples())}')

Graph built
Number of nodes: 464
Number of tripples: 466


Save the graph to disk

In [9]:
graph_path = Path(r'C:\Users\USER\rag_course\10b_graph_rag\data\graph_builtin.gml')
graph_path.parent.mkdir(parents=True, exist_ok=True)

graph.write_to_gml(str(graph_path))

print('Graph saved to:')
print(f'{graph_path}')

Graph saved to:
C:\Users\USER\rag_course\10b_graph_rag\data\graph_builtin.gml


Query the graph with GraphQAChain

In [10]:
from langchain_core.prompts import PromptTemplate

# Custom entity extractor: tells the chain what kinds of entities to look for
custom_extraction = PromptTemplate(
    template="""
    Extract all disease names, crop names, and pathogen names from this question.
    Return them as a comma-separated list. Use the exact case from the question.

    Question: {question}
    Entities:
    """,
    input_variables=["question"],
)

# Build the QA chain with our custom extractor
qa_chain = GraphQAChain.from_llm(
    llm=llm,
    graph=graph,
    entity_extraction_prompt=custom_extraction,   
    verbose=True,
)

print('✅ GraphQAChain ready with custom entity extractor')

# Test a bridge question
question = 'What do Malaria and Cassava Mosaic Virus have in common?'
answer = qa_chain.run(question)
print(f'\n🤖 Answer: {answer}')

✅ GraphQAChain ready with custom entity extractor


C:\Users\USER\AppData\Local\Temp\ipykernel_11736\906099511.py:27: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  answer = qa_chain.run(question)




> Entering new GraphQAChain chain...
Entities Extracted:
Malaria, Cassava Mosaic Virus
Full Context:
Malaria AFFECTS Nigerians
Malaria HAS_SYMPTOM Children_Dying_Of_Malaria
Malaria HAS_SYMPTOM Infant_Mortality
Malaria HAS_SYMPTOM Childhood_Mortality
Malaria HAS_SYMPTOM Maternal_Mortality
Malaria AFFECTS Millennium Development Goals
Malaria CAUSED_BY Under 5 Mortality
Malaria CAUSED_BY Childhood Mortality
Malaria CAUSED_BY Maternal Mortality
Malaria AFFECTS Children Aged Less Than 5 Years
Malaria AFFECTS Pregnant Women
Malaria AFFECTS Nigeria
Malaria IS_A Infectious Diseases
Cassava Mosaic Virus AFFECTS Cassava

> Finished chain.

🤖 Answer: Both Malaria and Cassava Mosaic Virus affect specific populations: Malaria affects Nigerians, particularly children under 5 years and pregnant women, while Cassava Mosaic Virus affects cassava plants. Additionally, both are associated with significant health and economic impacts in their respective contexts.


Inspect the graph

In [11]:
# Show the most connected entities
print('Sample triples from the graph:\n')

triples = graph.get_triples()

for t in triples[:15]:
    print(f'  {t[0]}  ->  {t[2]}  ({t[1]})')

Sample triples from the graph:

  Department Of Public Health  ->  IS_A  (Fmoh)
  Fmoh  ->  IS_A  (Public_Health_Department)
  Cerebral Spinal Meningitis  ->  AFFECTS  (Nigerians)
  Cholera  ->  AFFECTS  (Nigerians)
  Measles  ->  AFFECTS  (Nigerians)
  Lassa Fever  ->  AFFECTS  (Nigerians)
  Yellow Fever  ->  AFFECTS  (Nigerians)
  Diarrhoeal  ->  AFFECTS  (Nigerians)
  Malaria  ->  AFFECTS  (Nigerians)
  Malaria  ->  HAS_SYMPTOM  (Children_Dying_Of_Malaria)
  Malaria  ->  HAS_SYMPTOM  (Infant_Mortality)
  Malaria  ->  HAS_SYMPTOM  (Childhood_Mortality)
  Malaria  ->  HAS_SYMPTOM  (Maternal_Mortality)
  Malaria  ->  AFFECTS  (Millennium Development Goals)
  Malaria  ->  CAUSED_BY  (Under 5 Mortality)
